# Cleaning Data
OIBSIP · Data Analytics · Level 1 · Task 3

We clean the Kaggle **Titanic** dataset (`train.csv`, save it next to this notebook). It has missing values, inconsistent coding and outliers. Every decision is documented in markdown.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data quality report (before cleaning)

In [ ]:
raw = pd.read_csv("train.csv")
df = raw.copy()

def quality_report(d):
    return pd.Series({
        "rows": len(d),
        "columns": d.shape[1],
        "total nulls": int(d.isnull().sum().sum()),
        "duplicate rows": int(d.duplicated().sum()),
    })

before = quality_report(raw)
print(before, "\n")
print("Nulls per column:\n", raw.isnull().sum()[raw.isnull().sum() > 0], "\n")
print("Data types:\n", raw.dtypes)
raw.describe().T

## 2. Missing data
- **Age** (numeric, ~20% missing): fill with the **median** (robust to outliers).
- **Embarked** (categorical, 2 missing): fill with the **mode**.
- **Cabin** (~77% missing): too sparse to impute; **drop the column**, keeping a flag `HasCabin` instead.

In [ ]:
df["HasCabin"] = df["Cabin"].notna().astype(int)
df = df.drop(columns="Cabin")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
print("Nulls remaining:", df.isnull().sum().sum())

## 3. Duplicates

In [ ]:
dups = df.duplicated().sum()
df = df.drop_duplicates()
print("Duplicate rows removed:", dups)

## 4. Standardise inconsistent formatting
Standardise text case and expand coded values (`S/C/Q`) so categories are consistent and readable.

In [ ]:
df["Sex"] = df["Sex"].str.strip().str.title()
df["Embarked"] = df["Embarked"].str.strip().str.upper().map(
    {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"})
df["Name"] = df["Name"].str.strip()
print(df["Sex"].value_counts(), "\n")
print(df["Embarked"].value_counts())

## 5. Outlier detection (IQR method)
We flag values outside `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR`.

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "Fare"]:
    lo, hi = iqr_bounds(df[col])
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: bounds ({lo:.1f}, {hi:.1f}) -> {n} outliers")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
sns.boxplot(x=df["Age"], ax=axes[0]); sns.boxplot(x=df["Fare"], ax=axes[1])
plt.show()

**Decisions**
- **Fare:** extreme fares are real first-class tickets, not errors, but they skew the distribution, so we **cap (winsorise)** at the upper IQR bound in a new column `Fare_capped` and keep the original.
- **Age:** values flagged are plausible elderly passengers, so we **retain** them.

In [ ]:
lo, hi = iqr_bounds(df["Fare"])
df["Fare_capped"] = df["Fare"].clip(lower=lo, upper=hi)

## 6. Correct data types

In [ ]:
df["PassengerId"] = df["PassengerId"].astype(str)
df["Survived"] = df["Survived"].astype(int)
df["Pclass"] = df["Pclass"].astype("category")
df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")
df["Age"] = df["Age"].astype(float)
df["Fare"] = df["Fare"].astype(float)
df.dtypes

## 7. Before vs after

In [ ]:
after = quality_report(df)
summary = pd.DataFrame({"Before": before, "After": after})
summary.loc["object (text) columns"] = [raw.select_dtypes("object").shape[1], df.select_dtypes("object").shape[1]]
summary

## 8. Save the cleaned dataset

In [ ]:
df.to_csv("titanic_cleaned.csv", index=False)
print("Saved titanic_cleaned.csv with shape", df.shape)